# Dự báo giá đồng phiên kế tiếp bằng XGBoost

Notebook dùng `copper_8_indicators_2006_2026.csv` cùng thư mục. Mục tiêu là dự báo **Copper ở phiên giao dịch kế tiếp** từ thông tin đã có ở phiên hiện tại. Chia tập theo thời gian, không xáo trộn; validation nằm ở cuối train và chỉ dùng để chọn số cây bằng early stopping.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

SEED = 42
GPU_DEVICE = 'cuda:0'
gpu_probe = xgb.XGBRegressor(n_estimators=1, tree_method='hist', device=GPU_DEVICE)
gpu_probe.fit(np.array([[0.0], [1.0], [2.0], [3.0]]), np.array([0.0, 1.0, 2.0, 3.0]))
actual_device = json.loads(gpu_probe.get_booster().save_config())['learner']['generic_param']['device']
if actual_device != GPU_DEVICE:
    raise RuntimeError(f'XGBoost đang dùng {actual_device}, cần {GPU_DEVICE}. Kiểm tra bản XGBoost có CUDA và driver NVIDIA.')
print(f'XGBoost {xgb.__version__} đang dùng {actual_device}')
CSV_PATH = Path('copper_8_indicators_2006_2026.csv')
TRAIN_RATIOS = [0.90, 0.925, 0.95]
VALIDATION_FRACTION = 0.10
EARLY_STOPPING_ROUNDS = 60
FEATURE_COLUMNS = ['Copper', 'Copper_Volume', 'Dow_Jones', 'Nasdaq', 'Dollar_Index', 'TBill_10Y', 'Gold', 'Silver', 'WTI']

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy {CSV_PATH.resolve()}')
raw = pd.read_csv(CSV_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
missing = {'Date', *FEATURE_COLUMNS}.difference(raw.columns)
if missing:
    raise ValueError(f'Thiếu cột: {sorted(missing)}')
raw = raw[['Date', *FEATURE_COLUMNS]].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
if raw['Date'].duplicated().any() or not raw['Date'].is_monotonic_increasing:
    raise ValueError('Date phải duy nhất và tăng dần.')
print(f'{len(raw):,} phiên | {raw.Date.min().date()} → {raw.Date.max().date()} | XGBoost {xgb.__version__}')
raw.head()

XGBoost 3.4.1 đang dùng cuda:0
5,231 phiên | 2006-01-03 → 2026-10-29 | XGBoost 3.4.1


,Date,Copper,Copper_Volume,Dow_Jones,Nasdaq,Dollar_Index,TBill_10Y,Gold,Silver,WTI
0,2006-01-03,2.1680,682,10847.410156,2243.739990,89.839996,4.370,530.700012,9.087,63.139999
1,2006-01-04,2.2165,537,10880.150391,2263.459961,89.139999,4.356,533.900024,9.102,63.419998
2,2006-01-05,2.1385,554,10882.150391,2276.870117,89.330002,4.356,526.299988,8.809,62.790001
3,2006-01-06,2.1555,704,10959.309570,2305.620117,88.849998,4.379,539.700012,9.112,64.209999
4,2006-01-09,2.1670,286,11011.900391,2318.689941,89.250000,4.379,549.099976,9.222,63.500000


## Tạo đặc trưng dự báo

Mỗi dòng đặc trưng tại ngày *t* chỉ chứa giá trị quan sát đến ngày *t*; target là Copper tại ngày giao dịch kế tiếp. Các độ trễ và thống kê cuộn của đồng giúp mô hình học quán tính/xu hướng ngắn hạn. Các chỉ báo vĩ mô dùng giá trị hiện tại, độ trễ 1 và 5 phiên cùng thay đổi phần trăm. Các phép tính chỉ nhìn về quá khứ.

In [6]:
def make_supervised(data):
    f = pd.DataFrame(index=data.index)
    copper = data['Copper']
    for lag in [0, 1, 2, 3, 5, 10, 20]:
        f[f'Copper_lag_{lag}'] = copper.shift(lag)
    for window in [5, 10, 20]:
        f[f'Copper_roll_mean_{window}'] = copper.rolling(window).mean()
        f[f'Copper_roll_std_{window}'] = copper.rolling(window).std()
    macro = [c for c in FEATURE_COLUMNS if c not in ('Copper', 'Copper_Volume')]
    for col in macro:
        for lag in [0, 1, 5]:
            f[f'{col}_lag_{lag}'] = data[col].shift(lag)
        f[f'{col}_pct_1'] = data[col].pct_change(1)
        f[f'{col}_pct_5'] = data[col].pct_change(5)
    for lag in [0, 1, 5]:
        f[f'Copper_Volume_lag_{lag}'] = data['Copper_Volume'].shift(lag)
    dates = data['Date']
    f['month_sin'] = np.sin(2 * np.pi * dates.dt.month / 12)
    f['month_cos'] = np.cos(2 * np.pi * dates.dt.month / 12)
    f['dayofweek'] = dates.dt.dayofweek
    y = copper.shift(-1).rename('target_next_session')
    target_date = dates.shift(-1).rename('target_date')
    result = f.join(y).replace([np.inf, -np.inf], np.nan).dropna().copy()
    # Date mục tiêu chỉ dùng để lập báo cáo, không đưa vào feature của XGBoost.
    valid_index = result.index
    return result.drop(columns='target_next_session'), result['target_next_session'], target_date.loc[valid_index].reset_index(drop=True)

X, y, target_dates = make_supervised(raw)
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes):
    invalid = X.dtypes[[not pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes]]
    raise TypeError(f'XGBoost chỉ nhận feature số; dtype không hợp lệ: {invalid.to_dict()}')
print(f'{len(X):,} mẫu | {X.shape[1]} đặc trưng | mục tiêu: giá phiên kế tiếp')
X.head()

5,210 mẫu | 54 đặc trưng | mục tiêu: giá phiên kế tiếp


,Copper_lag_0,Copper_lag_1,Copper_lag_2,Copper_lag_3,Copper_lag_5,Copper_lag_10,Copper_lag_20,Copper_roll_mean_5,Copper_roll_std_5,Copper_roll_mean_10,...,WTI_lag_1,WTI_lag_5,WTI_pct_1,WTI_pct_5,Copper_Volume_lag_0,Copper_Volume_lag_1,Copper_Volume_lag_5,month_sin,month_cos,dayofweek
20,2.2590,2.2355,2.2345,2.2965,2.2185,2.1700,2.1680,2.2514,0.027505,2.20765,...,67.919998,65.849998,-0.020024,0.010782,319,344.0,269.0,0.866025,0.5,2
21,2.3185,2.2590,2.2355,2.2345,2.2315,2.1560,2.2165,2.2688,0.037479,2.22390,...,66.559998,66.260002,-0.028245,-0.023845,409,319.0,379.0,0.866025,0.5,3
22,2.3180,2.3185,2.2590,2.2355,2.2965,2.1300,2.1385,2.2731,0.042367,2.24270,...,64.680000,67.760002,0.010668,-0.035272,227,409.0,1159.0,0.866025,0.5,4
23,2.3365,2.3180,2.3185,2.2590,2.2345,2.1425,2.1555,2.2935,0.043671,2.26210,...,65.370003,68.349998,-0.003977,-0.047403,286,227.0,310.0,0.866025,0.5,0
24,2.2875,2.3365,2.3180,2.3185,2.2355,2.1725,2.1670,2.3039,0.030658,2.27360,...,65.110001,67.919998,-0.031024,-0.071113,191,286.0,344.0,0.866025,0.5,1


## Cấu hình và đánh giá

### Vì sao chọn các tham số này? (đọc trước khi train)

| Tham số | Giá trị khởi đầu | Lý do |
|---|---:|---|
| `objective` | `reg:squarederror` | Copper là target liên tục; tối ưu sai số bình phương. |
| `learning_rate` | `0.03` | Bước cập nhật vừa nhỏ để tổng hợp nhiều cây ổn định; tránh bước lớn dễ khớp quá nhanh. |
| `n_estimators` | `1200` tối đa | Cho phép đủ cây với learning rate nhỏ; **không nhất thiết dùng hết** vì early stopping dừng theo validation. |
| `early_stopping_rounds` | `60` | Chịu được dao động ngắn của RMSE validation, nhưng dừng nếu không cải thiện trong 60 vòng. |
| `max_depth` | `4` | Giới hạn tương tác và độ phức tạp; dataset chỉ khoảng 5.2 nghìn phiên nên cây quá sâu dễ học nhiễu. |
| `min_child_weight` | `5` | Yêu cầu khối lượng mẫu tối thiểu lớn hơn trước khi tách lá, giúp giảm các nhánh dựa trên ít phiên. |
| `subsample` | `0.8` | Mỗi cây dùng 80% hàng train để giảm tương quan giữa các cây và overfit. |
| `colsample_bytree` | `0.8` | Mỗi cây xem 80% đặc trưng; giảm lệ thuộc vào một biến giá/indicator. |
| `reg_alpha`, `reg_lambda` | `0.05`, `5.0` | L1/L2 regularization để phạt độ phức tạp; L2 cao vừa phải là lựa chọn thận trọng cho chuỗi nhỏ, nhiều đặc trưng trễ. |
| `tree_method`, `device` | `hist`, `cuda:0` | Huấn luyện histogram trên GPU NVIDIA đầu tiên. |
| `random_state` | `42` | Cố định tính ngẫu nhiên để lần chạy có thể lặp lại. |

Đây là **cấu hình khởi đầu có regularization**, không phải khẳng định đã tối ưu. Dùng 10% cuối của từng phần train làm validation để early stopping; test tương lai chỉ mở sau khi hoàn tất fit. Không xáo trộn chuỗi. XGBoost dạng cây không cần chuẩn hóa thang đo. Ba tỷ lệ train/test được giữ theo notebook tham khảo.

In [7]:
MODEL_PARAMS = dict(
    objective='reg:squarederror', eval_metric='rmse',
    n_estimators=1200, learning_rate=0.03, max_depth=4,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=5.0, gamma=0.0,
    tree_method='hist', device=GPU_DEVICE, random_state=SEED, n_jobs=-1,
)

def fit_one_ratio(ratio):
    n = len(X)
    split = int(n * ratio)
    val_start = int(split * (1 - VALIDATION_FRACTION))
    i_train = np.arange(0, val_start)
    i_val = np.arange(val_start, split)
    i_test = np.arange(split, n)
    model = xgb.XGBRegressor(**MODEL_PARAMS, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    model.fit(X.iloc[i_train], y.iloc[i_train], eval_set=[(X.iloc[i_val], y.iloc[i_val])], verbose=False)
    pred = model.predict(X.iloc[i_test])
    actual = y.iloc[i_test].to_numpy()
    persistence = X.iloc[i_test]['Copper_lag_0'].to_numpy()
    summary = {
        'train_ratio': ratio, 'n_train': len(i_train), 'n_validation': len(i_val), 'n_test': len(i_test),
        'test_start': target_dates.iloc[i_test].min().date().isoformat(),
        'test_end': target_dates.iloc[i_test].max().date().isoformat(),
        'best_iteration': int(model.best_iteration),
        'RMSE_USD_per_lb': float(np.sqrt(mean_squared_error(actual, pred))),
        'MAE_USD_per_lb': float(mean_absolute_error(actual, pred)),
        'MAPE_pct': float(mean_absolute_percentage_error(actual, pred) * 100),
        'persistence_RMSE_USD_per_lb': float(np.sqrt(mean_squared_error(actual, persistence))),
    }
    trace = pd.DataFrame({'Date': target_dates.iloc[i_test].to_numpy(), 'Actual': actual, 'XGBoost': pred, 'Persistence_Copper_t': persistence})
    return model, summary, trace

models, summaries, traces = {}, [], {}
for ratio in TRAIN_RATIOS:
    model, summary, trace = fit_one_ratio(ratio)
    models[ratio], traces[ratio] = model, trace
    summaries.append(summary)
    print(f"train={ratio:.1%} | best_iteration={summary['best_iteration']} | RMSE={summary['RMSE_USD_per_lb']:.5f} | MAE={summary['MAE_USD_per_lb']:.5f} | test={summary['test_start']}→{summary['test_end']}")
report = pd.DataFrame(summaries)
report

c:\Users\Acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\core.py:569: UserWarning: [00:52:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


train=90.0% | best_iteration=268 | RMSE=1.03944 | MAE=0.75556 | test=2024-10-02→2026-10-29
train=92.5% | best_iteration=424 | RMSE=1.19933 | MAE=0.97862 | test=2025-04-10→2026-10-29
train=95.0% | best_iteration=321 | RMSE=1.43492 | MAE=1.33406 | test=2025-10-16→2026-10-29


,train_ratio,n_train,n_validation,n_test,test_start,test_end,best_iteration,RMSE_USD_per_lb,MAE_USD_per_lb,MAPE_pct,persistence_RMSE_USD_per_lb
0,0.900,4220,469,521,2024-10-02,2026-10-29,268,1.039439,0.755558,12.566283,0.109474
1,0.925,4337,482,391,2025-04-10,2026-10-29,424,1.199332,0.978623,16.132639,0.118011
2,0.950,4454,495,261,2025-10-16,2026-10-29,321,1.434921,1.334061,21.646568,0.101627


## Grid search đầy đủ trên chuỗi thời gian

Cell dưới thử toàn bộ 9.216 tổ hợp tham số trong các khoảng đã đề xuất. Target là biến động một phiên `Copper(t+1) - Copper(t)`, sau đó được cộng lại vào giá hiện tại để tính RMSE theo USD/lb. Validation dùng 3 folds theo thứ tự thời gian; 10% dữ liệu cuối được giữ riêng làm test. Kết quả được ghi từng đợt vào CSV, vì vậy có thể chạy lại cell để tiếp tục khi bị ngắt.

In [8]:
from itertools import product
from sklearn.model_selection import TimeSeriesSplit

# Thêm đặc trưng tương đối để mô hình hoạt động tốt hơn khi giá chuyển sang vùng mới.
X_tuned = X.copy()
for lag in [1, 5, 10, 20]:
    X_tuned[f'Copper_return_{lag}'] = raw['Copper'].pct_change(lag).loc[X_tuned.index]
for window in [5, 20]:
    X_tuned[f'Copper_vs_ma_{window}'] = (raw['Copper'] / raw['Copper'].rolling(window).mean() - 1).loc[X_tuned.index]
    X_tuned[f'Copper_return_vol_{window}'] = raw['Copper'].pct_change().rolling(window).std().loc[X_tuned.index]
    X_tuned[f'Volume_vs_ma_{window}'] = (raw['Copper_Volume'] / raw['Copper_Volume'].rolling(window).mean() - 1).loc[X_tuned.index]
X_tuned = X_tuned.replace([np.inf, -np.inf], np.nan).dropna()
price_today = X_tuned['Copper_lag_0']
price_tomorrow = y.loc[X_tuned.index]
y_delta = price_tomorrow - price_today

PARAM_GRID = {
    'max_depth': [2, 3, 4, 5], 'min_child_weight': [5, 10, 20],
    'learning_rate': [0.01, 0.03, 0.05], 'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], 'reg_lambda': [1.0, 5.0, 10.0, 20.0],
    'reg_alpha': [0.0, 0.05, 0.5, 1.0],
}
param_names = list(PARAM_GRID)
candidates = [dict(zip(param_names, values)) for values in product(*(PARAM_GRID[k] for k in param_names))]
FINAL_TRAIN_RATIO = 0.90
final_split = int(len(X_tuned) * FINAL_TRAIN_RATIO)
X_search, y_search = X_tuned.iloc[:final_split], y_delta.iloc[:final_split]
search_splits = list(TimeSeriesSplit(n_splits=3).split(X_search))
SEARCH_FILE = Path('xgboost_grid_search_delta_cuda0.csv')
print(f'{len(candidates):,} cấu hình × 3 folds = {len(candidates) * 3:,} lần fit validation')

def score_candidate(params):
    fold_scores, fold_rounds = [], []
    for train_idx, valid_idx in search_splits:
        model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse',
            n_estimators=250, early_stopping_rounds=30, tree_method='hist', device=GPU_DEVICE,
            random_state=SEED, n_jobs=-1, **params)
        model.fit(X_search.iloc[train_idx], y_search.iloc[train_idx],
                  eval_set=[(X_search.iloc[valid_idx], y_search.iloc[valid_idx])], verbose=False)
        predicted_price = price_today.iloc[valid_idx].to_numpy() + model.predict(X_search.iloc[valid_idx])
        fold_scores.append(np.sqrt(mean_squared_error(price_tomorrow.iloc[valid_idx], predicted_price)))
        fold_rounds.append(model.best_iteration + 1)
    return np.mean(fold_scores), np.std(fold_scores), int(np.median(fold_rounds))

done = pd.read_csv(SEARCH_FILE) if SEARCH_FILE.exists() else pd.DataFrame()
done_keys = set(map(tuple, done[param_names].to_numpy())) if not done.empty else set()
pending, buffer = [p for p in candidates if tuple(p[k] for k in param_names) not in done_keys], []
for number, params in enumerate(pending, 1):
    mean_rmse, std_rmse, rounds = score_candidate(params)
    buffer.append({**params, 'validation_rmse_mean': mean_rmse, 'validation_rmse_std': std_rmse, 'best_round_median': rounds})
    if len(buffer) == 10 or number == len(pending):
        done = pd.concat([done, pd.DataFrame(buffer)], ignore_index=True)
        done.to_csv(SEARCH_FILE, index=False, encoding='utf-8-sig')
        print(f'Đã hoàn thành {len(done):,}/{len(candidates):,} cấu hình')
        buffer = []
grid_results = done.sort_values('validation_rmse_mean').reset_index(drop=True)
display(grid_results.head(10))

9,216 cấu hình × 3 folds = 27,648 lần fit validation
Đã hoàn thành 20/9,216 cấu hình
Đã hoàn thành 30/9,216 cấu hình
Đã hoàn thành 40/9,216 cấu hình
Đã hoàn thành 50/9,216 cấu hình
Đã hoàn thành 60/9,216 cấu hình
Đã hoàn thành 70/9,216 cấu hình
Đã hoàn thành 80/9,216 cấu hình
Đã hoàn thành 90/9,216 cấu hình
Đã hoàn thành 100/9,216 cấu hình
Đã hoàn thành 110/9,216 cấu hình
Đã hoàn thành 120/9,216 cấu hình
Đã hoàn thành 130/9,216 cấu hình
Đã hoàn thành 140/9,216 cấu hình
Đã hoàn thành 150/9,216 cấu hình
Đã hoàn thành 160/9,216 cấu hình
Đã hoàn thành 170/9,216 cấu hình
Đã hoàn thành 180/9,216 cấu hình
Đã hoàn thành 190/9,216 cấu hình
Đã hoàn thành 200/9,216 cấu hình
Đã hoàn thành 210/9,216 cấu hình
Đã hoàn thành 220/9,216 cấu hình
Đã hoàn thành 230/9,216 cấu hình
Đã hoàn thành 240/9,216 cấu hình
Đã hoàn thành 250/9,216 cấu hình
Đã hoàn thành 260/9,216 cấu hình
Đã hoàn thành 270/9,216 cấu hình
Đã hoàn thành 280/9,216 cấu hình
Đã hoàn thành 290/9,216 cấu hình
Đã hoàn thành 300/9,216 cấu hìn

KeyboardInterrupt: 

In [ ]:
# Refit train + validation bằng cấu hình thắng; test chỉ được dùng tại đây.
if len(grid_results) != len(candidates):
    raise RuntimeError('Grid search chưa hoàn tất; chạy lại cell trên để tiếp tục từ CSV.')
best_params = grid_results.loc[0, param_names].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_rounds = int(grid_results.loc[0, 'best_round_median'])
final_model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse',
    n_estimators=best_rounds, tree_method='hist', device=GPU_DEVICE, random_state=SEED, n_jobs=-1, **best_params)
final_model.fit(X_tuned.iloc[:final_split], y_delta.iloc[:final_split], verbose=False)
test_idx = np.arange(final_split, len(X_tuned))
predicted = price_today.iloc[test_idx].to_numpy() + final_model.predict(X_tuned.iloc[test_idx])
actual = price_tomorrow.iloc[test_idx].to_numpy()
persistence = price_today.iloc[test_idx].to_numpy()
best_metrics = pd.DataFrame([{'validation_RMSE_USD_per_lb': grid_results.loc[0, 'validation_rmse_mean'],
    'test_RMSE_USD_per_lb': np.sqrt(mean_squared_error(actual, predicted)),
    'test_MAE_USD_per_lb': mean_absolute_error(actual, predicted),
    'persistence_RMSE_USD_per_lb': np.sqrt(mean_squared_error(actual, persistence)), **best_params}])
best_metrics.to_csv('xgboost_best_delta_model_metrics.csv', index=False, encoding='utf-8-sig')
display(best_metrics)

In [ ]:
fig, axes = plt.subplots(len(TRAIN_RATIOS), 1, figsize=(14, 4 * len(TRAIN_RATIOS)), squeeze=False)
for ax, ratio in zip(axes[:, 0], TRAIN_RATIOS):
    trace = traces[ratio]
    ax.plot(trace['Date'], trace['Actual'], label='Giá thực tế', color='black', linewidth=1)
    ax.plot(trace['Date'], trace['XGBoost'], label='XGBoost', linewidth=1)
    ax.plot(trace['Date'], trace['Persistence_Copper_t'], label='Baseline: giá phiên hiện tại', alpha=0.7, linewidth=1)
    ax.set_title(f'Test {1-ratio:.1%} | dự báo copper phiên kế tiếp')
    ax.set_ylabel('USD/lb'); ax.grid(alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()

out_dir = Path.cwd()
report.to_csv(out_dir / 'xgboost_copper_metrics.csv', index=False, encoding='utf-8-sig')
for ratio, trace in traces.items():
    suffix = str(ratio).replace('.', '_')
    trace.to_csv(out_dir / f'xgboost_copper_predictions_{suffix}.csv', index=False, encoding='utf-8-sig')
print('Đã lưu xgboost_copper_metrics.csv và các file dự báo test.')

## Lưu ý diễn giải

Giá đồng trong CSV là giá hợp đồng tương lai Yahoo làm proxy. Kết quả là đánh giá dự báo một phiên kế tiếp trên các giai đoạn test cuối chuỗi; không đại diện cho lợi nhuận giao dịch sau phí. Các chỉ báo được giả định đã sẵn có tại thời điểm dự báo. Tham khảo cách chia thời gian trong `copper-lstm-sa_result.ipynb`; kết quả XGBoost không so sánh trực tiếp với MSE bài báo LSTM vì khác mô hình, dữ liệu và đơn vị metric.